In [1]:
# import libraries

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np

In [2]:
# load csv file 
# extracted the POS-volume and value columns from CBN's Financial Statistics sheet)

df = pd.read_csv("pos_data.csv")
df.columns = ["date", "volume", "value"]
df["volume"] = df["volume"].astype(str).str.replace(",", "").astype(float)
df["value"]  = df["value"].astype(str).str.replace(",", "").astype(float)
df["date"]   = pd.to_datetime(df["date"], format="%b-%y")
df           = df.sort_values("date").reset_index(drop=True)
df["avg_txn"] = (df["value"] * 1_000_000) / df["volume"]
df["year"]    = df["date"].dt.year
df["month"]   = df["date"].dt.month


In [3]:
# set style

PURPLE = "#534AB7"
TEAL   = "#1D9E75"
CORAL  = "#D85A30"
AMBER  = "#BA7517"
BG     = "#FAFAF8"
GRID   = "#E8E6DF"
TEXT   = "#2C2C2A"
MUTED  = "#888780"

plt.rcParams.update({
    "font.family":        "sans-serif",
    "font.size":          11,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.spines.left":   False,
    "axes.spines.bottom": False,
    "axes.grid":          True,
    "grid.color":         GRID,
    "grid.linewidth":     0.6,
    "axes.facecolor":     BG,
    "figure.facecolor":   BG,
    "text.color":         TEXT,
    "axes.labelcolor":    TEXT,
    "xtick.color":        MUTED,
    "ytick.color":        MUTED,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
})

def annotate_event(ax, x, label, color=CORAL, y_frac=0.90):
    ymin, ymax = ax.get_ylim()
    ypos = ymin + (ymax - ymin) * y_frac
    ax.axvline(pd.Timestamp(x), color=color, linewidth=1,
               linestyle="--", alpha=0.6)
    ax.text(pd.Timestamp(x), ypos, label, fontsize=7.5, color=color,
            rotation=90, va="top", ha="right", alpha=0.85)


In [4]:
# Creating chart 1: POS volume from 2009–2025 

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.fill_between(df["date"], df["volume"] / 1e9, alpha=0.12, color=PURPLE)
ax.plot(df["date"], df["volume"] / 1e9, color=PURPLE, linewidth=2)

ax.set_title("Nigeria POS transaction volume, 2009–2025",
             fontsize=14, fontweight="normal", pad=16, loc="left")
ax.set_ylabel("Transactions (billions)", color=MUTED, fontsize=9)
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x:.1f}B" if x >= 1 else f"{x*1000:.0f}M"))
ax.set_ylim(bottom=0)

events = [
    ("2012-01-01", "CBN cashless\npolicy launch"),
    ("2021-01-01", "eNaira &\nagent banking surge"),
    ("2023-02-01", "Naira redesign\ncash crunch"),
]
for dt, lbl in events:
    annotate_event(ax, dt, lbl)

peak_idx = df["volume"].idxmax()
ax.annotate(
    f"Peak: {df['volume'][peak_idx]/1e9:.2f}B\n({df['date'][peak_idx].strftime('%b %Y')})",
    xy=(df["date"][peak_idx], df["volume"][peak_idx] / 1e9),
    xytext=(pd.Timestamp("2024-01-01"), df["volume"][peak_idx] / 1e9 * 0.80),
    fontsize=8, color=PURPLE,
    arrowprops=dict(arrowstyle="->", color=PURPLE, lw=1),
    bbox=dict(boxstyle="round,pad=0.3", fc=BG, ec=PURPLE, lw=0.8))

ax.text(0.01, -0.08,
        "Source: Central Bank of Nigeria, Payment System Statistics (2009–2025)",
        transform=ax.transAxes, fontsize=7.5, color=MUTED)
plt.tight_layout()
plt.savefig("chart1_pos_volume.png", dpi=180, bbox_inches="tight")
plt.close()
print("Chart 1 created")


Chart 1 created


In [5]:
# Creating chart 2: Average Transaction Size

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.fill_between(df["date"], df["avg_txn"], alpha=0.12, color=TEAL)
ax.plot(df["date"], df["avg_txn"], color=TEAL, linewidth=2)

ax.set_title("Average POS transaction size, 2009–2025  (₦ per transaction)",
             fontsize=14, fontweight="normal", pad=16, loc="left")
ax.set_ylabel("₦ per transaction", color=MUTED, fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"₦{x:,.0f}"))
ax.set_ylim(bottom=0)

for dt, lbl in events:
    annotate_event(ax, dt, lbl)

# Start annotation
ax.annotate(
    f"₦{df['avg_txn'].iloc[0]:,.0f}\n(Jan 2009)",
    xy=(df["date"].iloc[0], df["avg_txn"].iloc[0]),
    xytext=(pd.Timestamp("2011-06-01"), df["avg_txn"].iloc[0] * 1.2),
    fontsize=8, color=TEAL,
    arrowprops=dict(arrowstyle="->", color=TEAL, lw=1),
    bbox=dict(boxstyle="round,pad=0.3", fc=BG, ec=TEAL, lw=0.8))


# End annotation 
ax.annotate(
    f"₦{df['avg_txn'].iloc[-1]:,.0f}\n(Jun 2025)",
    xy=(df["date"].iloc[-1], df["avg_txn"].iloc[-1]),
    xytext=(pd.Timestamp("2021-06-01"), df["avg_txn"].iloc[-1] * 1.8),
    fontsize=8, color=TEAL,
    arrowprops=dict(arrowstyle="->", color=TEAL, lw=1),
    bbox=dict(boxstyle="round,pad=0.3", fc=BG, ec=TEAL, lw=0.8))

# Source text 
ax.text(0.01, -0.08,
        "Source: Central Bank of Nigeria, Payment System Statistics (2009–2025)",
        transform=ax.transAxes, fontsize=7.5, color=MUTED)
plt.tight_layout()
plt.savefig("chart2_avg_transaction.png", dpi=180, bbox_inches="tight")
plt.close()
print("Chart 2 created")

Chart 2 created


In [6]:
# Creating chart 3: Year-on-Year growth
# Build YoY manually 

records = []
for i, row in df.iterrows():
    prev = df[(df["year"] == row["year"] - 1) & (df["month"] == row["month"])]
    if not prev.empty:
        vol_prev = prev["volume"].values[0]
        if vol_prev > 0:
            growth = (row["volume"] - vol_prev) / vol_prev * 100
            records.append({"date": row["date"], "yoy_growth": growth})

df_yoy = pd.DataFrame(records)

# Cap y-axis at 400% so the 2013 anomaly spike doesn't flatten every other bar. Annotate it as an outlier instead.
SPIKE_THRESHOLD = 400
spike_mask  = df_yoy["yoy_growth"] > SPIKE_THRESHOLD
normal_mask = ~spike_mask

bar_colors = [TEAL if g >= 0 else CORAL for g in df_yoy["yoy_growth"]]

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.bar(df_yoy["date"][normal_mask], df_yoy["yoy_growth"][normal_mask],
       width=25, color=[bar_colors[i] for i in df_yoy.index[normal_mask]], alpha=0.85)

# Draw clipped spike bars with hatching to show they are truncated
for i in df_yoy.index[spike_mask]:
    ax.bar(df_yoy["date"][i], SPIKE_THRESHOLD * 0.98, width=25,
           color=TEAL, alpha=0.5, hatch="//")
    ax.text(df_yoy["date"][i], SPIKE_THRESHOLD * 0.88,
            f"+{df_yoy['yoy_growth'][i]/100:.0f}x\n(truncated)",
            ha="center", va="top", fontsize=7, color=TEAL,
            bbox=dict(boxstyle="round,pad=0.2", fc=BG, ec=TEAL, lw=0.6))

ax.set_ylim(-50, SPIKE_THRESHOLD)
ax.axhline(0, color=MUTED, linewidth=0.8)
ax.set_title("Year-on-year POS volume growth (%), 2010–2025",
             fontsize=14, fontweight="normal", pad=16, loc="left")
ax.set_ylabel("YoY growth (%)", color=MUTED, fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))

# Annotate the real story bars
jan21 = df_yoy[df_yoy["date"] == pd.Timestamp("2021-01-01")]
if not jan21.empty:
    v = jan21["yoy_growth"].values[0]
    ax.annotate(f"+{v:.0f}%\n(Jan 2021)",
                xy=(jan21["date"].values[0], v),
                xytext=(pd.Timestamp("2019-06-01"), v + 40),
                fontsize=8, color=TEAL,
                arrowprops=dict(arrowstyle="->", color=TEAL, lw=1),
                bbox=dict(boxstyle="round,pad=0.3", fc=BG, ec=TEAL, lw=0.8))

mar23 = df_yoy[df_yoy["date"] == pd.Timestamp("2023-03-01")]
if not mar23.empty:
    v = mar23["yoy_growth"].values[0]
    ax.annotate(f"+{v:.0f}%\n(Mar 2023)",
                xy=(mar23["date"].values[0], min(v, SPIKE_THRESHOLD * 0.95)),
                xytext=(pd.Timestamp("2024-06-01"), 250),
                fontsize=8, color=TEAL,
                arrowprops=dict(arrowstyle="->", color=TEAL, lw=1),
                bbox=dict(boxstyle="round,pad=0.3", fc=BG, ec=TEAL, lw=0.8))

annotate_event(ax, "2021-01-01", "eNaira launch",  color=CORAL, y_frac=0.95)
annotate_event(ax, "2023-02-01", "Cash crunch",    color=CORAL, y_frac=0.95)

pos_patch = mpatches.Patch(color=TEAL,  alpha=0.85, label="Positive growth")
neg_patch = mpatches.Patch(color=CORAL, alpha=0.85, label="Negative growth")
ax.legend(handles=[pos_patch, neg_patch], fontsize=8, frameon=False)

ax.text(0.01, -0.08,
        "Source: Central Bank of Nigeria, Payment System Statistics (2009–2025)",
        transform=ax.transAxes, fontsize=7.5, color=MUTED)
plt.tight_layout()
plt.savefig("chart3_yoy_growth.png", dpi=180, bbox_inches="tight")
plt.close()
print("Chart 3 created")

Chart 3 created


In [7]:
# Creating chart 4: Overlay — POS volume vs Agent banking adoption 
# EFInA agent banking penetration (% of adults): 2016=~5%, 2018=~17%, 2020=24%, 2023=54%
# Source: EFInA Access to Finance Survey 2023

efina_dates  = [pd.Timestamp("2016-01-01"), pd.Timestamp("2018-01-01"),
                pd.Timestamp("2020-01-01"), pd.Timestamp("2023-01-01")]
efina_agent  = [5, 17, 24, 54]

fig, ax1 = plt.subplots(figsize=(12, 5.5))

# Left axis — POS volume
ax1.fill_between(df["date"], df["volume"] / 1e9, alpha=0.08, color=PURPLE)
ax1.plot(df["date"], df["volume"] / 1e9, color=PURPLE, linewidth=2,
         label="POS transaction volume (left axis)")
ax1.set_ylabel("POS transactions (billions)", color=PURPLE, fontsize=9)
ax1.tick_params(axis="y", labelcolor=PURPLE)
ax1.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x:.1f}B" if x >= 1 else f"{x*1000:.0f}M"))
ax1.set_ylim(bottom=0)

# Right axis — agent banking %
ax2 = ax1.twinx()
ax2.plot(efina_dates, efina_agent, color=TEAL, linewidth=2.5,
         linestyle="--", marker="o", markersize=7,
         label="Adults using financial agents, % (right axis)")
ax2.set_ylabel("Adults using financial agents (%)", color=TEAL, fontsize=9)
ax2.tick_params(axis="y", labelcolor=TEAL)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
ax2.set_ylim(0, 80)
ax2.spines["right"].set_visible(True)
ax2.spines["right"].set_color(GRID)

# Annotate the 2020 to 2023 agent surge
ax2.annotate("Agent banking\ndoubles: 24%→54%",
             xy=(pd.Timestamp("2023-01-01"), 54),
             xytext=(pd.Timestamp("2021-06-01"), 65),
             fontsize=8, color=TEAL,
             arrowprops=dict(arrowstyle="->", color=TEAL, lw=1),
             bbox=dict(boxstyle="round,pad=0.3", fc=BG, ec=TEAL, lw=0.8))

ax1.annotate("POS volume\ncrosses 1B/month",
             xy=(pd.Timestamp("2023-03-01"), 1.22),
             xytext=(pd.Timestamp("2021-01-01"), 1.1),
             fontsize=8, color=PURPLE,
             arrowprops=dict(arrowstyle="->", color=PURPLE, lw=1),
             bbox=dict(boxstyle="round,pad=0.3", fc=BG, ec=PURPLE, lw=0.8))

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2,
           fontsize=8, frameon=False, loc="upper left")

ax1.set_title(
    "Agent banking expansion tracks POS volume growth, 2016–2025",
    fontsize=14, fontweight="normal", pad=16, loc="left")

ax1.text(0.01, -0.08,
         "Sources: CBN Payment System Statistics (2009–2025); EFInA Access to Finance Survey 2023",
         transform=ax1.transAxes, fontsize=7.5, color=MUTED)

plt.tight_layout()
plt.savefig("chart4_agent_pos_overlay.png", dpi=180, bbox_inches="tight")
plt.close()
print("Chart 4 created")

Chart 4 created


In [10]:
# Creating chart 5: Financial Inclusion Funnel
# Shows the conversion gap from included to transactional to digital
# All figures are from EFInA A2F 2023

categories = [
    "Adult population\n(111 million)",
    "Financially\nincluded (74%)",
    "Formally\nincluded (64%)",
    "Used digital\npayments (45%)",
    "Receive income\ndigitally (14%)",
]
values    = [111, 82.1, 71.0, 49.9, 15.5]   # millions
bar_colors_funnel = [PURPLE, PURPLE, TEAL, TEAL, AMBER]

fig, ax = plt.subplots(figsize=(11, 5.5))
bars = ax.barh(categories, values, color=bar_colors_funnel, alpha=0.85, height=0.55)
ax.set_xlim(0, 130)
ax.set_xlabel("Million adults", color=MUTED, fontsize=9)
ax.invert_yaxis()

for bar, val in zip(bars, values):
    ax.text(val + 1.5, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}M", va="center", fontsize=9, color=TEXT)

# Gap annotation
ax.annotate("", xy=(15.5, 4), xytext=(49.9, 3),
            arrowprops=dict(arrowstyle="-", color=CORAL,
                            lw=1, linestyle="dashed"))
ax.text(33, 3.62, "34.4M people: digitally\ntransacting but cash-income",
        fontsize=7.5, color=CORAL, ha="center")

ax.set_title(
    "Nigeria's financial inclusion funnel — where conversion breaks down",
    fontsize=14, fontweight="normal", pad=16, loc="left")
ax.text(0.01, -0.08,
        "Source: EFInA Access to Finance Survey 2023",
        transform=ax.transAxes, fontsize=7.5, color=MUTED)

purple_patch = mpatches.Patch(color=PURPLE, alpha=0.85, label="Access metrics")
teal_patch   = mpatches.Patch(color=TEAL,   alpha=0.85, label="Digital payment metrics")
amber_patch  = mpatches.Patch(color=AMBER,  alpha=0.85, label="Deep integration metrics")
ax.legend(handles=[purple_patch, teal_patch, amber_patch],
          fontsize=8, frameon=False, loc="lower right")

plt.tight_layout()
plt.savefig("chart5_inclusion_funnel.png", dpi=180, bbox_inches="tight")
plt.close()
print("Chart 5 created.")

Chart 5 created.


In [11]:
# Key Stats

df = df.sort_values("date")  

start_vol = df["volume"].iloc[0]
end_vol   = df["volume"].iloc[-1]

start_avg = df["avg_txn"].iloc[0]
end_avg   = df["avg_txn"].iloc[-1]

print("\n KEY STATISTICS")
print("\n               ")
print(f"Jan 2009 volume:   {start_vol:>15,.0f} transactions")
print(f"Jun 2025 volume:   {end_vol:>15,.0f} transactions")
print(f"Total growth:      {end_vol/start_vol:>14,.0f}x increase")

print(f"\nJan 2009 avg txn:  ₦{start_avg:>12,.2f}")
print(f"Jun 2025 avg txn:  ₦{end_avg:>12,.2f}")

# Annual comparison
df["year"] = df["date"].dt.year

vol_2020 = df[df["year"] == 2020]["volume"].sum()
vol_2024 = df[df["year"] == 2024]["volume"].sum()

print(f"\n2020 annual volume: {vol_2020/1e9:.2f}B transactions")
print(f"2024 annual volume: {vol_2024/1e9:.2f}B transactions")
print(f"2020 to 2024 growth:   {vol_2024/vol_2020:.1f}x in 4 years")



 KEY STATISTICS

               
Jan 2009 volume:            89,349 transactions
Jun 2025 volume:     1,386,426,046 transactions
Total growth:              15,517x increase

Jan 2009 avg txn:  ₦   16,116.58
Jun 2025 avg txn:  ₦   18,508.54

2020 annual volume: 0.66B transactions
2024 annual volume: 13.08B transactions
2020 to 2024 growth:   19.9x in 4 years
